# ThreadMask v2: reproducible CPU experiment

Local predictions only; no Kaggle submission. Install requirements.txt, run from this repository, and provide authorized competition data. Full recorded tuning and evaluation outputs are in results/v2. Re-running commands reproduces an experiment; it does not restore a previously inspected test set to blind status.

In [ ]:
from pathlib import Path
import json, subprocess, sys
DATA = Path("MAGFiLO_1.0_Kaggle_2026")
assert DATA.exists(), "Supply authorized data first"

In [ ]:
subprocess.run([sys.executable,"-m","unittest","-v"],check=True)

The stored split was frozen before tuning: 35 development images and 100 prospective holdout images, grouped by observing day. The earlier 135-image set is excluded. The following cell checks that split reproducibly.

In [ ]:
subprocess.run([sys.executable,"experiment.py","freeze","--data",str(DATA)],check=True)

Optional reproduction: the next two cells rerun all 24 tuning trials, freeze the selected configuration, then evaluate both methods on the same held-out images. Do not adjust parameters after inspecting held-out outcomes.

In [ ]:
subprocess.run([sys.executable,"experiment.py","tune","--data",str(DATA)],check=True)

In [ ]:
subprocess.run([sys.executable,"experiment.py","evaluate","--data",str(DATA)],check=True)

In [ ]:
result=json.loads(Path("results/v2/blind-validation.json").read_text())
print(result["metric"])
print("Baseline:",result["baseline_mean_pq"])
print("V2:",result["v2_mean_pq"])

In [ ]:
subprocess.run([sys.executable,"predict_v2.py",str(DATA),"results/v2/candidate.csv"],check=True)
audit=json.loads(Path("results/v2/candidate.audit.json").read_text())
print("Test images:",len(audit["images"]))
print("Zero detections:",audit["zero_prediction_images"])

Each exported COCO RLE is decoded and compared pixel by pixel to its instance label. Review the organizer scorer, zero-detection image handling, data licenses, report template, and participant understanding before any submission. Nothing in this notebook registers or submits to Kaggle.

Public self-evaluation v6 uses pooled counts and all IoU > .5 pairs. The earlier displayed image-average diagnostic is retained as a different metric. This cell reproduces the public arithmetic audit, not the private Kaggle service.

In [ ]:
subprocess.run([sys.executable,"score_public.py",str(DATA)],check=True)
public_result=json.loads(Path("results/public-v6/evaluation.json").read_text())
print(public_result["results"])
print(public_result["scope"])